# 02 Results
Sensor selection, final test-set results, extrapolation check and report figures.

Run with **Run All**. The SVR sensor-selection step is slow (10–20 min), so it loads saved results unless `RERUN_SVR_SELECTION = True`.

In [ ]:
import sys
sys.path.insert(0, "../src")
from pathlib import Path
import pandas as pd

from degradation.data import load_dataset, split_inputs_target
from degradation.splitting import make_test_split, make_cv_folds
from degradation.models import build_models
from degradation.selection import forward_selection, evaluate_nested_subsets, condition_number
from degradation.evaluation import build_tuned_models, evaluate_on_test, extrapolation_test
from degradation import figures as F

RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)
FIGURES = Path("../figures"); FIGURES.mkdir(exist_ok=True)

X, y, dropped = split_inputs_target(load_dataset("../data/raw/data.txt"))
X_dev, X_test, y_dev, y_test = make_test_split(X, y)
tuned = build_tuned_models(build_models())
print("Condition number of the 12 standardised inputs:", round(condition_number(X_dev)))

## 1. Sensor selection
Forward selection with the decision tree ranks the sensors (fast). The tuned SVR is then scored on the first 1, 2, …, 12 sensors of that ranking.

In [ ]:
tree_sel = forward_selection(tuned["tree"], X_dev, y_dev, make_cv_folds())
tree_sel.to_csv(RESULTS / "tree_forward_selection.csv", index=False)
tree_sel.round(5)

In [ ]:
RERUN_SVR_SELECTION = False   # True = recompute (slow: 10-20 minutes)

if RERUN_SVR_SELECTION:
    svr_sel = evaluate_nested_subsets(tuned["svr"], X_dev, y_dev, make_cv_folds(), list(tree_sel["added"]))
    svr_sel.to_csv(RESULTS / "svr_nested_subsets.csv", index=False)
else:
    svr_sel = pd.read_csv(RESULTS / "svr_nested_subsets.csv")
svr_sel.round(5)

## 2. Final results on the locked test set
This is the only place the test set is used.

In [ ]:
test_table, predictions = evaluate_on_test(tuned, X_dev, y_dev, X_test, y_test)
test_table.to_csv(RESULTS / "test_set_results.csv")
pd.DataFrame({"true": y_test.values, **predictions}).to_csv(RESULTS / "test_set_predictions.csv", index=False)
test_table.round(6)

## 3. Extrapolation check
Each model is trained without one band of compressor decay and tested on that band.

In [ ]:
extrap = extrapolation_test({k: tuned[k] for k in ["linear", "tree", "svr"]}, X_dev, y_dev)
extrap.to_csv(RESULTS / "extrapolation.csv", index=False)
extrap.groupby(["model", "edge_band"])["mae"].mean().unstack().round(5)

## 4. Figures for the report

In [ ]:
F.plot_tree_tuning(pd.read_csv(RESULTS / "tree_tuning.csv"), FIGURES / "tree_tuning.png")
F.plot_svr_tuning(pd.read_csv(RESULTS / "svr_tuning.csv"), FIGURES / "svr_tuning.png")
F.plot_sensor_selection(tree_sel, svr_sel, FIGURES / "sensor_selection.png")
F.plot_predicted_vs_true(pd.read_csv(RESULTS / "test_set_predictions.csv"), FIGURES / "predicted_vs_true.png")
F.plot_extrapolation(extrap, FIGURES / "extrapolation.png");